# Sampler leave-one-source-out benchmark

Tests the legacy window-balanced and corrected participant-balanced samplers when an entire primary source is excluded from fitting. This measures whether the sampling correction improves real cross-source transport, not only pooled participant folds. No external frozen cohort is read.

In [1]:
from pathlib import Path
import sys,json,gc
import numpy as np,pandas as pd,torch
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from sklearn.metrics import roc_auc_score,balanced_accuracy_score,brier_score_loss
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name.lower()=='notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from models.stroke_gait_inception import StrokeGaitInception
P=ROOT/'data'/'processed'; D=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',D)
x=np.concatenate([np.load(P/'validated_acceleration_magnitude_windows_float32.npy'),np.load(P/'sint_maartenskliniek_external_windows_float32.npy')]); m=pd.concat([pd.read_csv(P/'validated_window_metadata.csv'),pd.read_csv(P/'sint_maartenskliniek_external_window_metadata.csv')],ignore_index=True); m=m[m.label.isin(['healthy','stroke'])].reset_index(drop=True); m['y']=m.label.eq('stroke').astype(int); m['source']=m.dataset_id; m['group']=m.participant_key.astype(str)
def weights(frame,mode):
    cells=frame.groupby(['source','y']).size(); den=np.asarray(pd.MultiIndex.from_frame(frame[['source','y']]).map(cells),float); w=1/den
    if mode=='participant_source_class':
        pc=frame.groupby('group').size(); cp=frame[['source','y','group']].drop_duplicates().groupby(['source','y']).size(); pden=np.asarray(pd.MultiIndex.from_frame(frame[['source','y']]).map(cp),float); w=frame.group.map(1/pc).to_numpy()/pden
    return torch.tensor(w,dtype=torch.double)
def evaluate(net,arr,meta,mean,std):
    with torch.inference_mode(): p=torch.sigmoid(net(torch.from_numpy(((arr-mean)/std).transpose(0,2,1).astype('float32')).to(D))).cpu().numpy()
    g=meta.assign(p=p).groupby(['group','y'],as_index=False).p.mean(); return {'participants':len(g),'healthy':int((g.y==0).sum()),'stroke':int((g.y==1).sum()),'auroc':roc_auc_score(g.y,g.p),'balanced_accuracy':balanced_accuracy_score(g.y,g.p>=.5),'healthy_specificity':float((g.loc[g.y==0,'p']<.5).mean()),'brier':brier_score_loss(g.y,g.p)}
rows=[]
for held in sorted(m.source.unique()):
    tr=m.source.ne(held).to_numpy(); va=~tr
    for seed in [42,137,202]:
        for mode in ['window_source_class','participant_source_class']:
            torch.manual_seed(seed); tx=x[tr]; mean,std=tx.reshape(-1,3).mean(0),tx.reshape(-1,3).std(0).clip(1e-4); z=torch.from_numpy(((tx-mean)/std).transpose(0,2,1).astype('float32')); y=torch.from_numpy(m.loc[tr,'y'].to_numpy('float32')); dl=DataLoader(TensorDataset(z,y),128,sampler=WeightedRandomSampler(weights(m.loc[tr],mode),len(z),replacement=True,generator=torch.Generator().manual_seed(seed+1000)))
            net=StrokeGaitInception().to(D); opt=torch.optim.AdamW(net.parameters(),1e-3,weight_decay=1e-4)
            for _ in range(8):
                net.train()
                for a,b in dl: opt.zero_grad(); loss=torch.nn.functional.binary_cross_entropy_with_logits(net(a.to(D)),b.to(D)); loss.backward(); opt.step()
            net.eval(); rows.append({'held_out_source':held,'seed':seed,'mode':mode,**evaluate(net,x[va],m.loc[va],mean,std)}); del net,opt,dl,z,y; gc.collect(); torch.cuda.empty_cache() if D.type=='cuda' else None; print('complete',held,seed,mode)
out=pd.DataFrame(rows); out.to_csv(P/'sampler_leave_one_source_out_benchmark.csv',index=False); print(out.groupby(['held_out_source','mode'])[['auroc','balanced_accuracy','healthy_specificity','brier']].agg(['mean','std']).round(4))

device: cuda


complete felius_2024 42 window_source_class


complete felius_2024 42 participant_source_class


complete felius_2024 137 window_source_class


complete felius_2024 137 participant_source_class


...

complete voisard_2025 42 participant_source_class


complete voisard_2025 137 window_source_class


complete voisard_2025 137 participant_source_class


complete voisard_2025 202 window_source_class


complete voisard_2025 202 participant_source_class
                                                auroc          \
                                                 mean     std   
held_out_source      mode                                       
felius_2024          participant_source_class  0.8806  0.0037   
                     window_source_class       0.8792  0.0058   
sint_maartenskliniek participant_source_class  0.8550  0.0050   
                     window_source_class       0.8900  0.0218   
voisard_2025         participant_source_class  0.8605  0.0893   
                     window_source_class       0.9303  0.0296   

                                              balanced_accuracy          \
                                                           mean     std   
held_out_source      mode                                                 
felius_2024          participant_source_class            0.7766  0.0234   
                     window_source_class                 0.7582

Do not replace the development sampler on pooled results alone. It must be non-inferior on the worst held-out source, particularly healthy specificity and calibration. This is still internal-source transport evidence, not clinical external validation.